# Zero-shot image classification (15 points)

In [1]:
import json
import torch
import os
from PIL import Image
from tqdm import tqdm
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import classification_report

## Zero-shot image classification
### Build Dataset

In [2]:
from google.colab import drive
drive.mount("/content/drive")
# !tar -xvf /content/drive/MyDrive/test_set_hw4.tar -C /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
with open("/content/drive/MyDrive/imagenet-simple-labels.json") as f:
    labels_1000 = json.load(f)

synset_to_label_1000 = {}
synset_to_id_1000 = {}
labels_1000 = []

mapping_file = "/content/drive/MyDrive/LOC_synset_mapping.txt"
with open(mapping_file, "r") as f:
    for idx, line in enumerate(f):
        synset, text = line.strip().split(" ", 1)
        synset_to_label_1000[synset] = text.split(",")[0]
        synset_to_id_1000[synset] = idx
        labels_1000.append(text.split(",")[0])


ROOT = "/content/drive/MyDrive/train"

train_synsets = sorted([
    s for s in os.listdir(ROOT)
    if os.path.isdir(os.path.join(ROOT, s))
])

print("train classes:", len(train_synsets))
print(train_synsets[:5])

labels = [synset_to_label_1000[s] for s in train_synsets]

num_classes = len(labels)

print("num_classes:", num_classes)  # 64
print(labels[:5])

train classes: 64
['n01532829', 'n01558993', 'n01704323', 'n01749939', 'n01770081']
num_classes: 64
['house finch', 'robin', 'triceratops', 'green mamba', 'harvestman']


### Initialize CLIP's vision transformer and text tokenizer from the HuggingFace library.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
def predict_best_prompt(image, text_prompts):
    """
    Args:
        image: PIL.Image 或 numpy array
        text_prompts: List[str]，prompts

    Returns:
        best_idx: int, pred ide
        best_prompt: str, pred_c
        probs: torch.Tensor, (1, N)
    """
    inputs = processor(
        text=text_prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.inference_mode():
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image   # (1, N_prompts)
        probs = logits_per_image.softmax(dim=1)       # (1, N_prompts)

    best_idx = probs.argmax(dim=1).item()
    best_prompt = text_prompts[best_idx]
    return best_idx, best_prompt, probs


### Perform zero-shot classification on the mini-Imagenet dataset using the following prompts.

In [ ]:

PROMPTS = [
    "a blurry photo of a {}.",
    "a photo of a {}.",
    "a bad photo of the {}.",
    "a photo of the large {}.",
    "a photo of the small {}.",
    "itap of a {}."
]

ROOT = "/content/drive/MyDrive/train"
n_per_folder = 100

def extract_num(filename):
    num_part = filename[9:].split(".")[0]
    return int(num_part)


for p_idx, prompt_template in enumerate(PROMPTS):
    y_true = []
    y_pred = []

    print(f"\n Prompt {p_idx + 1} / {len(PROMPTS)}")
    print("Template:", prompt_template)

    text_prompts = [prompt_template.format(label) for label in labels]

    for synset in tqdm(os.listdir(ROOT)):
        folder = os.path.join(ROOT, synset)

        true_id = train_synsets.index(synset)  # true label id

        imgs = [f for f in os.listdir(folder) if f.endswith(".jpg")]
        imgs.sort(key=extract_num)  # sort

        for img_name in imgs[:n_per_folder]:
            img_path = os.path.join(folder, img_name)
            image = Image.open(img_path).convert("RGB")

            pred_id, _, _ = predict_best_prompt(image, text_prompts)

            y_true.append(true_id)  # class id
            y_pred.append(pred_id) # pred id

    target_names = labels
    print(classification_report(
        y_true, y_pred,
        # labels=list(range(num_classes)),
        target_names=target_names,
        # zero_division=0
    ))




 Prompt 1 / 6
Template: a blurry photo of a {}.


100%|██████████| 64/64 [06:19<00:00,  5.92s/it]


                  precision    recall  f1-score   support

     house finch       0.80      0.90      0.85       100
           robin       0.95      0.77      0.85       100
     triceratops       0.92      0.94      0.93       100
     green mamba       0.71      0.97      0.82       100
      harvestman       0.72      0.93      0.81       100
          toucan       0.90      0.99      0.94       100
       jellyfish       0.99      0.87      0.93       100
          dugong       0.87      0.97      0.92       100
    Walker hound       0.61      0.97      0.75       100
          Saluki       0.86      0.77      0.81       100
   Gordon setter       0.62      0.87      0.72       100
        komondor       0.88      0.87      0.87       100
           boxer       0.93      0.65      0.76       100
 Tibetan mastiff       0.52      0.79      0.62       100
  French bulldog       0.90      0.66      0.76       100
    Newfoundland       0.78      0.18      0.29       100
miniature poo

100%|██████████| 64/64 [06:07<00:00,  5.75s/it]


                  precision    recall  f1-score   support

     house finch       0.79      0.90      0.84       100
           robin       0.96      0.77      0.86       100
     triceratops       0.96      0.85      0.90       100
     green mamba       0.74      0.98      0.84       100
      harvestman       0.73      0.91      0.81       100
          toucan       0.88      0.99      0.93       100
       jellyfish       0.98      0.86      0.91       100
          dugong       0.86      0.98      0.92       100
    Walker hound       0.57      0.96      0.72       100
          Saluki       0.89      0.66      0.76       100
   Gordon setter       0.51      0.91      0.66       100
        komondor       0.91      0.77      0.83       100
           boxer       0.94      0.59      0.72       100
 Tibetan mastiff       0.78      0.57      0.66       100
  French bulldog       0.88      0.73      0.80       100
    Newfoundland       0.76      0.41      0.53       100
miniature poo

...

                  precision    recall  f1-score   support

     house finch       0.75      0.91      0.82       100
           robin       0.95      0.78      0.86       100
     triceratops       0.99      0.82      0.90       100
     green mamba       0.69      0.98      0.81       100
      harvestman       0.74      0.91      0.82       100
          toucan       0.84      0.99      0.91       100
       jellyfish       0.84      0.94      0.89       100
          dugong       0.79      0.96      0.87       100
    Walker hound       0.66      0.92      0.77       100
          Saluki       0.84      0.77      0.80       100
   Gordon setter       0.55      0.90      0.68       100
        komondor       0.94      0.59      0.72       100
           boxer       0.94      0.60      0.73       100
 Tibetan mastiff       0.71      0.60      0.65       100
  French bulldog       0.85      0.74      0.79       100
    Newfoundland       0.76      0.32      0.45       100
miniature poo

100%|██████████| 64/64 [06:18<00:00,  5.91s/it]


                  precision    recall  f1-score   support

     house finch       0.80      0.87      0.83       100
           robin       0.94      0.76      0.84       100
     triceratops       0.94      0.91      0.92       100
     green mamba       0.72      0.97      0.83       100
      harvestman       0.74      0.93      0.83       100
          toucan       0.84      0.99      0.91       100
       jellyfish       0.91      0.92      0.92       100
          dugong       0.74      0.98      0.84       100
    Walker hound       0.58      0.94      0.72       100
          Saluki       0.79      0.77      0.78       100
   Gordon setter       0.48      0.92      0.63       100
        komondor       0.90      0.82      0.86       100
           boxer       0.98      0.51      0.67       100
 Tibetan mastiff       0.63      0.64      0.63       100
  French bulldog       0.86      0.71      0.78       100
    Newfoundland       0.73      0.19      0.30       100
miniature poo

100%|██████████| 64/64 [06:05<00:00,  5.72s/it]

                  precision    recall  f1-score   support

     house finch       0.81      0.90      0.85       100
           robin       0.94      0.77      0.85       100
     triceratops       0.99      0.84      0.91       100
     green mamba       0.71      0.98      0.82       100
      harvestman       0.70      0.92      0.79       100
          toucan       0.77      0.99      0.86       100
       jellyfish       0.99      0.75      0.85       100
          dugong       0.59      0.98      0.74       100
    Walker hound       0.60      0.93      0.73       100
          Saluki       0.81      0.60      0.69       100
   Gordon setter       0.40      0.93      0.56       100
        komondor       0.86      0.87      0.87       100
           boxer       1.00      0.41      0.58       100
 Tibetan mastiff       0.49      0.72      0.59       100
  French bulldog       0.90      0.66      0.76       100
    Newfoundland       0.79      0.23      0.36       100
miniature poo

### Look at the classification reports, write a short (i.e. a paragraph at most) analysis noting any surprising conclusions or findings. Why do you think you got those results?

The prediction model achieved 72%-76% accuracy on classicication, which is robust on different prompt templates. Descriptive modifiers for the photo prompt do not significantly affect model performance, whereas it is sensitive to the Descriptive modifiers for the object. The model is more sensitive to changes in scene related descriptors, leading to a more pronounced performance degradation.

# Prompt Tuning using Context Optimization (CoOp) (20 points)

### upload learned prompt

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_path_4 = "/content/drive/MyDrive/to_gdrive/vit_b32_ep50_16shots/nctx4_cscFalse_ctpend/seed1/prompt_learner/model.pth.tar-50"
ckpt_path_16 = "/content/drive/MyDrive/to_gdrive/vit_b32_ep50_16shots/nctx16_cscFalse_ctpend/seed1/prompt_learner/model.pth.tar-50"

state_4 = torch.load(ckpt_path_4, map_location="cpu", weights_only=False)
state_16 = torch.load(ckpt_path_16, map_location="cpu", weights_only=False)

print("Top-level keys:")
print(state_4.keys())
print(state_16.keys())

promp_4 = state_4["state_dict"]
promp_16 = state_16["state_dict"]
print("state_dict keys:")
print(promp_4.keys())

print("Loaded successfully!")

ctx_4 = promp_4["ctx"].to(device)  # Learned prompt tokens
token_prefix_4 = promp_4["token_prefix"].to(device)  # [num_classes, prefix_len, 512]
token_suffix_4 = promp_4["token_suffix"].to(device)

ctx_16 = promp_16["ctx"]
token_prefix_16 = promp_16["token_prefix"]
token_suffix_16 = promp_16["token_suffix"]

print("ctx 4 shape:", ctx_4.shape)  # [4, 512] or [16, 512]
print("token_prefix 4", token_prefix_4.shape)
print("token_suffix 4", token_suffix_4.shape)
print("ctx 16 shape:", ctx_16.shape)  # [4, 512] or [16, 512]
print("token_prefix 16", token_prefix_16.shape)
print("token_suffix 16", token_suffix_16.shape)

Top-level keys:
dict_keys(['state_dict', 'epoch', 'optimizer', 'scheduler'])
dict_keys(['state_dict', 'epoch', 'optimizer', 'scheduler'])
state_dict keys:
odict_keys(['ctx', 'token_prefix', 'token_suffix'])
Loaded successfully!
ctx 4 shape: torch.Size([4, 512])
token_prefix 4 torch.Size([1000, 1, 512])
token_suffix 4 torch.Size([1000, 72, 512])
ctx 16 shape: torch.Size([16, 512])
token_prefix 16 torch.Size([1000, 1, 512])
token_suffix 16 torch.Size([1000, 60, 512])


In [ ]:
# !pip install git+https://github.com/openai/CLIP.git

In [18]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

token_embedding = clip_model.token_embedding
pos_embed = clip_model.positional_embedding
text_proj = clip_model.text_projection
transformer = clip_model.transformer


def build_learned_prompt_embedding(ctx, prefix, suffix, trained_id):
    """
    ctx:    [M, 512]
    prefix: [C, P, 512]
    suffix: [C, S, 512]
    return: [C, L, 512],  L = P+M+S
    """
    prefix = prefix[trained_id]   # [C_sub, P, d]
    suffix = suffix[trained_id]   # [C_sub, S, d]

    C_sub, P, d = prefix.shape
    M = ctx.shape[0]
    _, S, _ = suffix.shape

    ctx_expand = ctx.unsqueeze(0).expand(C_sub, M, d)  # [C_sub, M, d]
    prompts = torch.cat([prefix, ctx_expand, suffix], dim=1)

    return prompts


def encode_prompts_with_clip(prompts: torch.Tensor):
    C, L, d = prompts.shape
    dtype = clip_model.dtype

    pos = pos_embed[:L].unsqueeze(0).to(device, dtype)
    x = prompts.to(device, dtype) + pos  # [C,L,d]

    # shape to [L,C,d] for transformer
    x = x.permute(1, 0, 2)

    with torch.no_grad():
        x = transformer(x)   # [L,C,d]

    # back to [C,L,d]
    x = x.permute(1, 0, 2)

    # EOS token (last)
    x = x[:, -1, :]

    x = x @ text_proj.to(device, dtype)
    x = x / x.norm(dim=-1, keepdim=True)

    return x.float()


def coop_predict(image, ctx, prefix, suffix, trained_id):
    # image feature
    img = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        img_feat = clip_model.encode_image(img)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)  # [1,d]

    # text feature (learned prefix+ctx+suffix)
    prompts = build_learned_prompt_embedding(ctx, prefix, suffix, trained_id)
    text_feats = encode_prompts_with_clip(prompts)  # [C,d]

    # make dtype consistent
    text_feats = text_feats.to(img_feat.dtype)

    sims = img_feat @ text_feats.T  # [1,C]
    pred_id = sims.argmax(dim=1).item()
    return pred_id


### build dataset

In [9]:
train_synsets = sorted([
  s for s in os.listdir(ROOT)
  if os.path.isdir(os.path.join(ROOT, s))
  ])

trained_id = [synset_to_id_1000[synset] for synset in train_synsets] ## trained id
labels = [synset_to_label_1000[synset] for synset in train_synsets]


In [10]:
def extract_num(filename):
    num_part = filename[9:].split(".")[0]
    return int(num_part)

n_per_folder = 100

y_true = []
y_pred = []

for synset in tqdm(os.listdir(ROOT)):
  folder = os.path.join(ROOT, synset)

  true_id = train_synsets.index(synset)  # true label id

  imgs = [f for f in os.listdir(folder) if f.endswith(".jpg")]
  imgs.sort(key=extract_num)  # sort

  for img_name in imgs[:n_per_folder]:
    img_path = os.path.join(folder, img_name)
    image = Image.open(img_path).convert("RGB")

    pred_id = coop_predict(image, ctx_4, token_prefix_4, token_suffix_4, trained_id)

    y_true.append(true_id)  # class id
    y_pred.append(pred_id) # pred id


100%|██████████| 64/64 [07:39<00:00,  7.18s/it]


In [11]:
print(f"Prompt with 4 ctx")
print(classification_report(
    y_true, y_pred,
    target_names=labels
))

Prompt with 4 ctx
                  precision    recall  f1-score   support

     house finch       0.78      0.31      0.44       100
           robin       0.53      0.90      0.66       100
     triceratops       0.98      0.88      0.93       100
     green mamba       0.92      0.76      0.83       100
      harvestman       0.82      0.66      0.73       100
          toucan       0.81      0.92      0.86       100
       jellyfish       0.69      0.93      0.79       100
          dugong       0.96      0.81      0.88       100
    Walker hound       0.59      0.79      0.67       100
          Saluki       0.67      0.02      0.04       100
   Gordon setter       0.70      0.07      0.13       100
        komondor       1.00      0.12      0.21       100
           boxer       1.00      0.08      0.15       100
 Tibetan mastiff       0.75      0.03      0.06       100
  French bulldog       0.60      0.70      0.65       100
    Newfoundland       0.15      0.88      0.26      

In [19]:
y_true = []
y_pred = []

for synset in tqdm(os.listdir(ROOT)):
  folder = os.path.join(ROOT, synset)

  true_id = train_synsets.index(synset)  # true label id

  imgs = [f for f in os.listdir(folder) if f.endswith(".jpg")]
  imgs.sort(key=extract_num)  # sort

  for img_name in imgs[:n_per_folder]:
    img_path = os.path.join(folder, img_name)
    image = Image.open(img_path).convert("RGB")

    pred_id = coop_predict(image, ctx_16, token_prefix_16, token_suffix_16, trained_id)

    y_true.append(true_id)  # class id
    y_pred.append(pred_id) # pred id

print(f"Prompt with 16 ctx")
print(classification_report(
    y_true, y_pred,
    target_names=labels
))

100%|██████████| 64/64 [07:28<00:00,  7.00s/it]

Prompt with 16 ctx
                  precision    recall  f1-score   support

     house finch       0.98      0.50      0.66       100
           robin       0.64      0.89      0.74       100
     triceratops       0.71      0.96      0.82       100
     green mamba       0.93      0.84      0.88       100
      harvestman       0.95      0.57      0.71       100
          toucan       0.80      0.99      0.88       100
       jellyfish       0.91      0.90      0.90       100
          dugong       0.95      0.80      0.87       100
    Walker hound       0.48      0.93      0.63       100
          Saluki       0.80      0.08      0.15       100
   Gordon setter       0.65      0.60      0.62       100
        komondor       0.72      0.87      0.79       100
           boxer       0.86      0.24      0.38       100
 Tibetan mastiff       0.52      0.63      0.57       100
  French bulldog       0.67      0.62      0.65       100
    Newfoundland       0.37      0.70      0.48     

### 4. Write a short (i.e. a paragraph at most) analysis of the results compared to Part 1's.

Longer shared learned context inproves the classification performance. Compared to part 1's results, the learned prompts provide stable prediction results, while part1's results variate across different prompt templates. Additionally, the learned prompts achieved higher precision, indicating the model maked fewer false positive and more likely to be correct to predict a class.